In [123]:
!pip install dm-haiku optax chex gymnasium

import copy
from shutil import rmtree
import random
import collections
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
from IPython.display import HTML
from base64 import b64encode
import chex
import warnings
from collections import namedtuple
warnings.filterwarnings('ignore')

Упражнение 1

In [124]:
def linear_policy(parameters, observation):
    dot_value = jnp.dot(parameters, observation)
    chosen_action = jax.lax.select( dot_value > 0, 1,0)
    return chosen_action

In [125]:
test_obs = jnp.array([1.0, 1.0, 2.0, 4.0])
params_pos = jnp.array([1.0, 1.0, 1.0, 1.0])
params_neg = jnp.array([-1.0, -1.0, -1.0, -1.0])
print(f"Положительные параметры -> действие: {linear_policy(params_pos, test_obs)}")
print(f"Отрицательные параметры -> действие: {linear_policy(params_neg, test_obs)}")

Положительные параметры -> действие: 1
Отрицательные параметры -> действие: 0


Упражнение 2

In [126]:
def run_episode(environment):
    episode_return = 0
    game_over = False
    policy_weights = jnp.array([1, -2, 2, -1])
    current_obs, _ = environment.reset()
    while not game_over:
        action_taken = linear_policy(policy_weights, current_obs)
        action_taken = np.array(action_taken)

        current_obs, reward, terminated, truncated, _ = environment.step(action_taken)
        episode_return += reward
        game_over = terminated or truncated

    return episode_return

In [127]:
test_env = gym.make("CartPole-v1")
episode_score = run_episode(test_env)
print(f"Суммарная награда за эпизод: {episode_score}")
test_env.close()

Суммарная награда за эпизод: 21.0


Случайный поиск политики (RPS)

Упражнение 3

In [128]:
RandomPolicySearchParams = namedtuple("RandomPolicySearchParams", ["current", "best"])

In [129]:
def rps_choose_action(key, params, actor_state, obs, evaluation=False):
    best_action = linear_policy(params.best, obs)
    current_action = linear_policy(params.current, obs)

    selected_action = jax.lax.select( evaluation, best_action, current_action   )
    return selected_action, actor_state

Упражнение 4

In [130]:
def get_new_random_weights(random_key, old_weights, min_val=-2.0, max_val=2.0):
    weights_shape = old_weights.shape
    weights_type = old_weights.dtype

    new_weights = jax.random.uniform( random_key, shape=weights_shape, dtype=weights_type, minval=min_val, maxval=max_val)
    return new_weights

In [131]:
test_old = np.ones(4, dtype=np.float32)
test_key = jax.random.PRNGKey(42)
new_w = get_new_random_weights(test_key, test_old)
print(f"Новые веса: {new_w}")

Новые веса: [-0.04516172  0.7191887   0.46508598  0.24406433]


Упражнение 5

In [132]:
RandomPolicyLearnState = namedtuple( "RandomPolicyLearnState", ["best_avg_return"])

In [133]:
def rps_learn(random_key, params, learn_state, memory_sample):
    best_weights = params.best
    current_weights = params.current

    current_avg_return = memory_sample
    best_avg_return = learn_state.best_avg_return

    best_weights = jax.lax.select( current_avg_return > best_avg_return, current_weights, best_weights)

    best_avg_return = jax.lax.select( current_avg_return > best_avg_return, current_avg_return, best_avg_return)
    fresh_weights = get_new_random_weights(random_key, current_weights)

    updated_params = RandomPolicySearchParams( current=fresh_weights, best=best_weights)
    updated_learn_state = RandomPolicyLearnState(best_avg_return)

    return updated_params, updated_learn_state

Градиенты политики (Policy Gradients, PG)

Упражнение 6

In [134]:
def compute_weighted_log_prob(action_probability, episode_return):
    log_prob_value = jnp.log(action_probability)
    weighted_result = log_prob_value * episode_return
    return weighted_result

In [135]:
test_prob = 0.8
test_return = 100
wlp_result = compute_weighted_log_prob(test_prob, test_return)
print(f"log(0.8) * 100 = {wlp_result}")

log(0.8) * 100 = -22.314353942871094


Упражнение 7

In [136]:
def compute_rewards_to_go(reward_list):
    rewards_to_go = []
    running_sum = 0
    for r in reversed(reward_list):
        running_sum += r
        rewards_to_go.append(running_sum)
    rewards_to_go.reverse()
    return rewards_to_go

In [137]:
test_rewards = [1, 2, 3, 4]
rtg = compute_rewards_to_go(test_rewards)
print(f"Rewards: {test_rewards}")
print(f"Rewards-to-go: {rtg}")

Rewards: [1, 2, 3, 4]
Rewards-to-go: [10, 9, 7, 4]


Упражнение 8

In [138]:
def sample_action(random_number, logits_vector):
    sampled_action = jax.random.categorical(random_number, logits_vector)
    return sampled_action

In [139]:
sample_key = jax.random.PRNGKey(42)
sample_logits = np.array([1.0, 2.0])
sampled = sample_action(sample_key, sample_logits)
print(f"Логиты: {sample_logits}")
print(f"Выбранное действие: {sampled}")

Логиты: [1. 2.]
Выбранное действие: 1


Упражнение 9

In [140]:
def policy_gradient_loss(action_id, logits_vec, reward_to_go):
    action_probs = jax.nn.softmax(logits_vec)
    chosen_action_prob = action_probs[action_id]
    weighted_log = compute_weighted_log_prob(chosen_action_prob, reward_to_go)
    loss_value = -weighted_log
    return loss_value

In [141]:
test_logits = np.array([1.0, 2.0])
pg_loss = policy_gradient_loss(1, test_logits, 10)
print(f"Loss: {pg_loss}")

Loss: 3.1326165199279785


Q-Learning

Упражнение 10

In [142]:
def select_greedy_action(q_value_vector):
    best_action_idx = jnp.argmax(q_value_vector)
    return best_action_idx

In [143]:
test_q = jnp.array([1.0, 1.0, 3.0, 4.0])
greedy = select_greedy_action(test_q)
print(f"Q-значения: {test_q}")
print(f"Жадное действие: {greedy}")

Q-значения: [1. 1. 3. 4.]
Жадное действие: 3


Упражнение 11

In [144]:
def compute_squared_error(predicted_val, target_val):
    squared_diff = jnp.square(predicted_val - target_val)
    return squared_diff

In [145]:
sq_err = compute_squared_error(1.0, 4.0)
print(f"Квадрат разности (1-4)^2 = {sq_err}")

Квадрат разности (1-4)^2 = 9.0


Упражнение 12

In [146]:
def compute_bellman_target(reward_val, is_terminal, next_state_q):
    bellman_target = jax.lax.select( is_terminal == 1.0, reward_val, reward_val + jnp.max(next_state_q))
    return bellman_target

In [147]:
next_q = np.array([3.0, 2.0])
bt1 = compute_bellman_target(1.0, 0.0, next_q)
bt2 = compute_bellman_target(1.0, 1.0, next_q)
print(f"Не терминальное: {bt1}")
print(f"Терминальное: {bt2}")

Не терминальное: 4.0
Терминальное: 1.0


Упражнение 13

In [148]:
def q_learning_loss(q_values, action_taken, reward_val, is_done, next_state_q):
    chosen_q = q_values[action_taken]
    bellman_target = compute_bellman_target(reward_val, is_done, next_state_q)
    loss_val = compute_squared_error(chosen_q, bellman_target)
    return loss_val

In [149]:
test_qvals = np.array([3.0, 2.0])
q_loss = q_learning_loss(test_qvals, 1, 2.0, 0.0, np.array([3.0, 2.0]))
print(f"Loss: {q_loss}")

Loss: 9.0


Упражнение 14

In [150]:
def select_random_action(rand_key, total_actions):
    random_action_id = jax.random.randint(rand_key, shape=(), minval=0, maxval=total_actions)
    return random_action_id

In [158]:
k1 = jax.random.PRNGKey(6)
k2 = jax.random.PRNGKey(1000)
rand1 = select_random_action(k1, 2)
rand2 = select_random_action(k2, 2)
print(f"Случайное действие (6): {rand1}")
print(f"Случайное действие (1000): {rand2}")

Случайное действие (6): 0
Случайное действие (1000): 0


Упражнение 15

In [152]:
EPSILON_DECAY_TIMESTEPS = 3000
EPSILON_MIN_value = 0.1

In [153]:
def get_epsilon(num_timesteps):
    epsilon_val = 1.0 - (num_timesteps / EPSILON_DECAY_TIMESTEPS) * (1.0 - EPSILON_MIN_value)
    epsilon_val = jax.lax.select( epsilon_val < EPSILON_MIN_value, EPSILON_MIN_value, epsilon_val)
    return epsilon_val

Упражнение 16

In [154]:
def select_epsilon_greedy_action(rand_key, q_value_vec, timestep_count):
    num_actions = len(q_value_vec)
    epsilon_val = get_epsilon(timestep_count)
    random_draw = jax.random.uniform(rand_key)
    is_exploring = random_draw < epsilon_val
    final_action = jax.lax.select( is_exploring, select_random_action(rand_key, num_actions), select_greedy_action(q_value_vec))

    return final_action